# Aprendizado de Máquina — Lista prática 04

## Métodos Não Paramétricos (KNN)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O Nadaraya--Watson cabe em duas linhas de `numpy`, e escrevê-las é a melhor forma
de entender o que ele faz. Depois disso a lista mede, sobre 300 amostras, o viés
de fronteira que a nota descreve — e o resultado tem uma reviravolta:

> **a regressão linear local reduz o viés na borda pela metade, como a teoria
> promete. Se isso a torna o método melhor ali depende da janela.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.neighbors import KNeighborsRegressor

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — Nadaraya--Watson à mão

O estimador é uma média ponderada de **todas** as observações,

$$\widehat r(x) = \sum_{i=1}^n w_i(x)\,y_i,
  \qquad w_i(x) = \frac{K\!\left(\frac{x - x_i}{h}\right)}
                       {\sum_{j=1}^n K\!\left(\frac{x - x_j}{h}\right)}.$$

Escreva os três núcleos e o estimador. Repare que a função recebe uma **grade**
de pontos de uma vez: `x0[:, None] - x_tr[None, :]` monta a matriz de todas as
diferenças, de tamanho (grade × treino).

In [ ]:
def nucleo_uniforme(u):
    return ...                      # (a)


def nucleo_gaussiano(u):
    return ...                               # (b)


def nucleo_epanechnikov(u):
    return ...                           # (c)


def nadaraya_watson(x_tr, y_tr, x0, h, nucleo=nucleo_gaussiano):
    W = nucleo((x0[:, None] - x_tr[None, :]) / h)                # pesos nao normalizados
    soma = W.sum(axis=1)
    # onde nenhum ponto de treino entrou na janela, o estimador nao existe (0/0)
    return np.where(soma > 0, (...) / np.where(soma > 0, soma, 1), np.nan)   # (d)

A amostra é a de sempre: $r(x)=\operatorname{sen}(1{,}5x)+0{,}3x$ com ruído
$N(0;\,0{,}7^2)$, $n=50$, semente 2026.

In [ ]:
def r(x):
    return np.sin(1.5 * x) + 0.3 * x


A, B, SIGMA, N_TR = -3.0, 3.0, 0.7, 50

rng = np.random.default_rng(2026)
x_tr = rng.uniform(A, B, size=N_TR)
y_tr = r(x_tr) + rng.normal(0, SIGMA, size=N_TR)
grade = np.linspace(A, B, 300)

for nome, K in [("uniforme", nucleo_uniforme),
                ("gaussiano", nucleo_gaussiano),
                ("Epanechnikov", nucleo_epanechnikov)]:
    est = nadaraya_watson(x_tr, y_tr, grade, h=0.4, nucleo=K)
    ok = ~np.isnan(est)
    eqm = np.mean((est[ok] - r(grade[ok])) ** 2)
    print(f"{nome:14s} EQM contra r: {eqm:.4f}   pontos sem vizinho: {int((~ok).sum())}")

> **Sua vez.** Desenhe as três curvas ajustadas sobre a nuvem, junto com $r$
> verdadeira. Onde estão os buracos dos núcleos de suporte compacto?

---
## Exercício 2 — a janela $h$ é que decide

Agora varie $h$ e escolha o melhor por validação cruzada de 5 dobras — a
ferramenta da Aula 03 aplicada a um método que não tem `.fit()`.

In [ ]:
hs = np.array([0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.2, 2.0])
cv = skm.KFold(5, shuffle=True, random_state=2026)
eqm_cv = []

for h in hs:
    erros = []
    for i_tr, i_te in cv.split(x_tr):
        pred = nadaraya_watson(..., ..., x_tr[i_te], h)   # (a) ajuste SEM a dobra
        erros.append(np.mean((y_tr[i_te] - pred) ** 2))
    eqm_cv.append(np.mean(erros))

eqm_cv = np.array(eqm_cv)
h_melhor = hs[...]                          # (b)

for h, e in zip(hs, eqm_cv):
    print(f"h = {h:4.2f}:  EQM (CV) {e:.4f}")
print(f"\nmelhor h por CV: {h_melhor}")

A validação cruzada escolheu $h$ **sem** conhecer $r$. Como aqui nós conhecemos,
dá para conferir se ela acertou: meça o erro contra a $r$ verdadeira na grade.

In [ ]:
est = nadaraya_watson(x_tr, y_tr, grade, h_melhor)
print(f"EQM contra r verdadeira, com h = {h_melhor}: {...:.4f}")   # (a)

---
## Exercício 3 — KNN na mesma amostra

O KNN fixa o **número** de vizinhos e deixa o raio variar; o Nadaraya--Watson faz
o contrário. Escolha o $k$ por CV, com as mesmas dobras, e compare os dois no
mesmo pé.

In [ ]:
X_tr = x_tr.reshape(-1, 1)
ks = np.arange(1, 26)

eqm_knn = np.array([
    -skm.cross_val_score(KNeighborsRegressor(n_neighbors=...),   # (a)
                         X_tr, y_tr, cv=cv,
                         scoring="neg_mean_squared_error").mean()
    for k in ks
])

k_melhor = ks[...]                         # (b)
print(f"melhor k por CV: {k_melhor}  (EQM {eqm_knn.min():.4f})")

knn = KNeighborsRegressor(n_neighbors=k_melhor).fit(X_tr, y_tr)
eqm_r = np.mean((knn.predict(grade.reshape(-1, 1)) - r(grade)) ** 2)
print(f"EQM contra r verdadeira: {eqm_r:.4f}")

---
## Exercício 4 — o viés de fronteira, medido

Aqui está o exercício central da aula, e ele **não pode ser feito numa amostra
só**: viés é uma média sobre amostras, e numa única realização o ruído domina.

Vamos comparar o Nadaraya--Watson com a **regressão linear local**, que em cada
ponto ajusta uma reta ponderada em vez de uma constante. Complete a montagem do
sistema de mínimos quadrados ponderados.

In [ ]:
def linear_local(x_tr, y_tr, x0, h):
    """Ajusta uma reta ponderada em torno de cada ponto de x0 e devolve o intercepto."""
    saida = np.empty(len(x0))
    for j, ponto in enumerate(x0):
        w = nucleo_gaussiano((ponto - x_tr) / h)
        # centramos em 'ponto', para que o intercepto SEJA a predicao ali
        Z = np.column_stack([np.ones_like(x_tr), ...])          # (a)
        A_ = Z.T @ (w[:, None] * Z)
        b_ = Z.T @ (...)                                            # (b)
        saida[j] = np.linalg.solve(A_, b_)[...]                            # (c) o intercepto
    return saida

Agora repita 300 vezes: sorteie uma amostra nova, estime nos pontos de interesse
com os dois métodos, e guarde. O viés é a média das estimativas menos o valor
verdadeiro.

In [ ]:
pontos = np.array([-3.0, 0.0, 3.0])       # duas fronteiras e o centro
REPETICOES = 300

for h in (0.4, 0.8):
    rng = np.random.default_rng(2026)
    est_nw = np.zeros((REPETICOES, len(pontos)))
    est_ll = np.zeros((REPETICOES, len(pontos)))

    for b in range(REPETICOES):
        xb = rng.uniform(A, B, size=N_TR)
        yb = r(xb) + rng.normal(0, SIGMA, size=N_TR)
        est_nw[b] = nadaraya_watson(xb, yb, pontos, h)
        est_ll[b] = linear_local(xb, yb, pontos, h)

    print(f"===== h = {h} =====")
    print("  x0     metodo    vies    variancia     EQM")
    for j, x0 in enumerate(pontos):
        for nome, est in (("NW", est_nw), ("LL", est_ll)):
            vies = ...                    # (a)
            variancia = ...                        # (b)
            print(f"  {x0:5.1f}   {nome}    {vies:+7.4f}   {variancia:8.4f}   "
                  f"{...:8.4f}")             # (c) a decomposicao da Aula 01
    print()

**Responda** na célula abaixo, como comentário: olhando a coluna do EQM, qual dos
dois métodos você usaria na fronteira? A resposta é a mesma para $h=0{,}4$ e para
$h=0{,}8$?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o núcleo quase não muda o EQM (0,063 a 0,132), mas os de suporte compacto deixam 10 pontos sem estimativa |
| 2 | a janela muda o EQM de CV de 0,64 a 1,09; a CV escolhe $h=0{,}30$ |
| 3 | NW com $h=0{,}30$ bate KNN com $k=5$ contra $r$: 0,0726 contra 0,1236 |
| 4 | na fronteira o viés do NW é 6 a 10 vezes o do linear local — e cresce com $h$, enquanto o do linear local não |
| 4 | ainda assim, em $h=0{,}4$ o EQM do linear local na fronteira é **3× pior**, por variância |

**A seguir.** A Aula 05 explica com teoria por que todo método de vizinhança
degrada quando $p$ cresce — e o Exercício 4 já deu a pista, porque em dimensão
alta quase todo ponto é ponto de fronteira.